<div style="
  background: linear-gradient(135deg, #667eea, #764ba2, #ff9a9e);
  padding: 32px;
  border-radius: 24px;
  text-align: center;
  color: #ffffff;
  box-shadow: 0 8px 20px rgba(0,0,0,0.15);
  margin-bottom: 20px;
">
  <h1 style="font-size: 42px; margin-bottom: 8px;">🎲 Bayesian Probability Simulation 🔬</h1>
  <h2 style="font-size: 24px; margin-top: 0; color: #f0e6ff;">From Conditional Probability to Naive Bayes Classification</h2>
  <p style="font-size: 18px; line-height: 1.7; color: #f0e6ff;">
    A refactored, simulation-driven exploration of Bayesian inference, Monte Carlo convergence,<br>
    and text classification — with corrected theory and reproducible R code.
  </p>
  <div style="background-color: rgba(255,255,255,0.18); display: inline-block; padding: 10px 18px;
              border-radius: 18px; margin-top: 12px; font-size: 16px; color: #ffffff;">
    R &nbsp;•&nbsp; Bayes' Theorem &nbsp;•&nbsp; Law of Total Probability &nbsp;•&nbsp; Monte Carlo &nbsp;•&nbsp; Naive Bayes
  </div>
</div>

### Notebook Overview

This notebook refactors the original course assignment into a **question-free, explanation-driven** format. Each section presents the **analytical derivation** followed immediately by a **Monte Carlo simulation** that validates it.

| Part | Topic | Core Concept |
|------|-------|--------------|
| 1.1 | Airport Security Scanner | Law of Total Probability & Bayes' theorem, simulation convergence |
| 1.2 | Fraud Detection | Bayes' theorem with one and two conditionally independent features |
| 2.1 | Monty Hall Problem | Conditional probability, switching vs staying |
| 2.2 | Infinite Monkey Theorem | Rare events, geometric / binomial intuition, `1-(1-p)^N` |
| 3 | Spam Email Classification | Bernoulli Naive Bayes from scratch, data leakage avoidance, Laplace smoothing |

**Reproducibility:** `set.seed(42)` (or `123`) is set before every stochastic block. Analytical values are printed alongside empirical estimates.

**Changes vs original:** Fixed deterministic-shuffle simulation (`rep`+`sample` → `rbinom`), wrong `84/113` → `84/133`, binary `any()` 0/1 bug in monkey convergence, data-leakage in spam DTM, Laplace denominator `+V` → `+2`, plus all convergence plots now use replicated runs with confidence bands.

In [ ]:
# Global reproducibility and libraries
set.seed(42)
# Required libraries (install once externally: install.packages(c("tm","SnowballC","e1071","caret")))
library(tm)
library(SnowballC)
library(e1071)
# library(caret)  # not needed — manual stratified sampling used


<div style="
  background: linear-gradient(135deg, #74c69d, #48cae4);
  padding: 26px;
  border-radius: 22px;
  margin-top: 28px;
  border: 4px solid #2d6a4f;
  box-shadow: 0 8px 22px rgba(0, 0, 0, 0.18);
  color: #081c15;
">
  <h1 style="color: #081c15; background-color: rgba(255,255,255,0.65); padding: 12px 16px; border-radius: 14px; margin-top: 0;">
    📡 Part 1 — Bayesian Reasoning with Simulation
  </h1>
  <p style="font-size: 17px; line-height: 1.7; color: #133f30; font-weight: 500;">
    Intuition and simulation of fundamental probability concepts: Law of Total Probability, Bayes' theorem,
    and how empirical frequencies converge to theoretical values as predicted by the Law of Large Numbers.
  </p>
</div>

<div style="
  background: linear-gradient(135deg, #a8edea, #fed6e3);
  padding: 24px;
  border-radius: 22px;
  margin-top: 28px;
  border: 4px solid #8e6fa0;
  box-shadow: 0 8px 20px rgba(0,0,0,0.12);
  color: #1a1a2e;
">
  <h1 style="color: #2b0a2a; background-color: rgba(255,255,255,0.55); padding: 10px 16px; border-radius: 14px; margin-top: 0; font-size: 26px;">
    ✈️ 1.1 Airport Security System — Law of Total Probability & Conditional Probability
  </h1>
  <p style="font-size: 16px; line-height: 1.7; color: #2b0a2a;">
    Scenario: an airport scanner detects prohibited objects. We derive <code>P(Alarm)</code> and
    <code>P(Object | Alarm)</code> and validate them with a Monte Carlo experiment.
  </p>
</div>

#### 1.1.1 Analytical Derivation

Given:

* $P(\text{Object}) = 0.05$
* $P(\text{Alarm} \mid \text{Object}) = 0.90$
* $P(\text{Alarm} \mid \neg\text{Object}) = 0.08$

**Law of Total Probability:**

$$P(\text{Alarm}) = P(\text{Alarm}\mid\text{Object})P(\text{Object}) + P(\text{Alarm}\mid\neg\text{Object})P(\neg\text{Object})$$

$$= 0.90 \times 0.05 + 0.08 \times 0.95 = 0.045 + 0.076 = 0.121$$

**Bayes' theorem:**

$$P(\text{Object}\mid\text{Alarm}) = \frac{P(\text{Alarm}\mid\text{Object})P(\text{Object})}{P(\text{Alarm})} = \frac{0.045}{0.121} = \frac{45}{121} \approx 0.3719$$

Note $P(\neg\text{Object}) = 1 - P(\text{Object}) = 0.95$ is used explicitly.


In [ ]:
# 1.1 — Analytical calculation (code implements the formulas)
p_object <- 0.05
p_alarm_given_object <- 0.90
p_alarm_given_no_object <- 0.08

p_alarm <- p_alarm_given_object * p_object + p_alarm_given_no_object * (1 - p_object)
p_object_given_alarm <- (p_alarm_given_object * p_object) / p_alarm

cat(sprintf("P(Alarm) = %.4f\n", p_alarm))
cat(sprintf("P(Object|Alarm) = %.4f  (45/121 = %.4f)\n", p_object_given_alarm, 45/121))


#### 1.1.2 Monte Carlo Simulation (Corrected)

**Fix vs original:** The original used `c(rep(1, N*p_object), rep(0, ...))` then `sample()`, which forces exactly $N\cdot p$ objects (zero variance, and breaks for non-integer $N\cdot p$ e.g. $N=10$). The correct Monte Carlo draws each passenger independently:

* `object_status ~ Bernoulli(p_object)`  via `rbinom(N, 1, p_object)`
* `alarm | object ~ Bernoulli(p_alarm_given_object)` and `alarm | \neg object ~ Bernoulli(p_alarm_given_no_object)` via `rbinom` per group.

This preserves $\text{Var}(\text{count}) = Np(1-p)$ and satisfies the Law of Large Numbers.


In [ ]:
# 1.1 — Simulation with correct Bernoulli draws (N = 100,000)
set.seed(42)
N <- 100000

# Each passenger independently has object with prob p_object
object_status <- rbinom(N, 1, p_object)

alarm <- integer(N)
idx_obj <- which(object_status == 1)
idx_no_obj <- which(object_status == 0)

alarm[idx_obj] <- rbinom(length(idx_obj), 1, p_alarm_given_object)
alarm[idx_no_obj] <- rbinom(length(idx_no_obj), 1, p_alarm_given_no_object)

emp_p_alarm <- mean(alarm == 1)
# Guard against zero alarms (NaN)
emp_p_object_given_alarm <- if (sum(alarm == 1) == 0) NA else mean(object_status[alarm == 1] == 1)

cat(sprintf("Empirical P(Alarm)         = %.4f (theoretical %.4f)\n", emp_p_alarm, p_alarm))
cat(sprintf("Empirical P(Object|Alarm)  = %.4f (theoretical %.4f)\n", emp_p_object_given_alarm, p_object_given_alarm))


#### 1.1.3 Analytical vs Empirical Comparison


In [ ]:
# 1.1 — Bar chart: analytical vs simulation
png("results/airport_comparison.png", width=900, height=700, res=120)
par(mar=c(5,5,4,2))
bar_positions <- barplot(
  height = rbind(c(p_alarm, p_object_given_alarm), c(emp_p_alarm, emp_p_object_given_alarm)),
  beside = TRUE,
  names.arg = c("P(Alarm)", "P(Object|Alarm)"),
  col = c("skyblue", "orange"),
  ylim = c(0, max(c(p_alarm, p_object_given_alarm, emp_p_alarm, emp_p_object_given_alarm), na.rm=TRUE) * 1.3),
  main = "Airport: Analytical vs Simulation (N = 100,000)",
  ylab = "Probability",
  border = "black",
  cex.names = 1.1, cex.lab = 1.1
)
legend("topright", legend = c("Analytical", "Simulation"), fill = c("skyblue", "orange"), bty = "n", cex=1.05)
grid(nx = NA, ny = NULL, col = "gray", lty = "dotted")
# annotate values
text(bar_positions, rbind(c(p_alarm, p_object_given_alarm), c(emp_p_alarm, emp_p_object_given_alarm)) + 0.015,
     labels = sprintf("%.3f", rbind(c(p_alarm, p_object_given_alarm), c(emp_p_alarm, emp_p_object_given_alarm))),
     cex=0.85)
dev.off()

bar_positions <- barplot(
  height = rbind(c(p_alarm, p_object_given_alarm), c(emp_p_alarm, emp_p_object_given_alarm)),
  beside = TRUE,
  names.arg = c("P(Alarm)", "P(Object|Alarm)"),
  col = c("skyblue", "orange"),
  ylim = c(0, max(c(p_alarm, p_object_given_alarm, emp_p_alarm, emp_p_object_given_alarm), na.rm=TRUE) * 1.3),
  main = "Airport: Analytical vs Simulation (N = 100,000)",
  ylab = "Probability",
  border = "black"
)
legend("topright", legend = c("Analytical", "Simulation"), fill = c("skyblue", "orange"), bty = "n")
grid(nx = NA, ny = NULL, col = "gray", lty = "dotted")


#### 1.1.4 Convergence Analysis

**Fix vs original:** Single run per $N$ has high variance (especially $N=10$ where $E[\text{alarm}]\approx1.2$ may be zero → `NaN`). The corrected version **replicates each $N$ $B=300$ times**, then plots the **mean ± SD** band. Convergence to the red theoretical line demonstrates the Law of Large Numbers.
